In [1]:
import pandas as pd
import numpy as np
import scipy as sp
import sklearn
import importlib
import databox as db
import joblib
from hashlib import md5
# import sample_group_stats as sgst
from sklearn.mixture import BayesianGaussianMixture
from sklearn.decomposition import FastICA
import extools as ex
import scipy.stats as st
import scipy.special as sp

OK -- read the ontology db (19571, 4)
Restored 303734 tags and 191240 tokens. Run update_corpus_tagdata() to update.
-- loaded the skipgram model tank/fasttext_8.model --


In [2]:


cedf = pd.read_parquet('data/q_sep/CE-ICA_data_8-dim.pq')


cedf

0         1         2  \
category   token_id                                                    
topic      Eur-987c_25_vole-3          -0.666552  0.022111  1.186840   
taxongroup Sci-501e_7_squirrel-5        0.437882 -0.811388  0.390039   
topic      Aur-441a_64_jellyfish-30    -0.026720 -0.699322 -0.699815   
           Cli-c809_26_animal-15       -0.409737 -0.535458  0.564966   
           Blab943_11_bird-4           -0.917884 -0.693017 -0.027334   
...                                          ...       ...       ...   
           Sem-88a0_194_Hymenoptera-27  0.037199 -0.702670 -1.493168   
taxongroup Rep-aa64_32_animals-26      -0.711836  0.719764 -0.809127   
           Rep-aa64_32_animals-26      -0.711836  0.719764 -0.809127   
           Rep-aa64_32_animals-26      -0.711836  0.719764 -0.809127   
           Pap-75c9_71_Papilio-42       1.411378 -0.681664  1.200986   

                                               3         4         5  \
category   token_id                                                    
topic      Eur-987c_25_vole-3           0.275075 -0.343453 -0.190398   
taxongroup Sci-501e_7_squirrel-5        0.792916 -0.113640 -1.519081   
topic      Aur-441a_64_jellyfish-30    -0.574663 -0.736788  0.411543   
           Cli-c809_26_animal-15        1.442520  0.227825 -0.542453   
           Blab943_11_bird-4           -0.217798  0.241200 -0.264970   
...                                          ...       ...       ...   
           Sem-88a0_194_Hymenoptera-27 -0.015171  4.126123  0.032796   
taxongroup Rep-aa64_32_animals-26       0.375698 -0.843446  0.428591   
           Rep-aa64_32_animals-26       0.375698 -0.843446  0.428591   
           Rep-aa64_32_animals-26       0.375698 -0.843446  0.428591   
           Pap-75c9_71_Papilio-42       1.385699 -2.137870  0.232634   

                                               6         7  
category   token_id                                         
topic      Eur-987c_25_vole-3          -0.318431  0.340504  
taxongroup Sci-501e_7_squirrel-5       -0.657930  1.530297  
topic      Aur-441a_64_jellyfish-30    -0.603104  0.550438  
           Cli-c809_26_animal-15        0.014104 -0.415746  
           Blab943_11_bird-4           -0.315390  1.291464  
...                                          ...       ...  
           Sem-88a0_194_Hymenoptera-27 -0.065156  0.388183  
taxongroup Rep-aa64_32_animals-26      -0.062365  0.129428  
           Rep-aa64_32_animals-26      -0.062365  0.129428  
           Rep-aa64_32_animals-26      -0.062365  0.129428  
           Pap-75c9_71_Papilio-42       0.051112  0.597190  

[54011 rows x 8 columns]

In [3]:
from som_lattice import SomLattice



In [4]:
som_fn = 'data/SOM8.pkl'
train = False

som8 = SomLattice(8)

if train:
    som8.train_SOM(np.array(cedf))
    joblib.dump(som8.SOM,som_fn)
else:
    som8.SOM = joblib.load('data/SOM8.pkl')
    
print(som8.SOM.shape)

(20, 16, 8)


In [5]:
BMUs = []
for i in range(len(cedf)):
    BMUs.append(som8.find_BMU(np.array(cedf.iloc[i])))
    if len(BMUs) % 10000 == 0:
        print(len(BMUs))

10000
20000
30000
40000
50000


In [6]:
bmu_df = pd.DataFrame(BMUs, columns=['x','y'], index=cedf.index)
bmu_df.sample(5)

x   y
category   token_id                         
topic      Tri-09f5_14_trilobites-11   8   6
           Ept-9841_15_Eptesicus-8     7   3
           Ani-4d8b_177_parrot-15      9   3
           Rhi-fb33_2_ungulates-67     7  12
taxongroup Chi-c10a_287_chimpanzee-2  17   3

In [7]:
bmu_bins = bmu_df.reset_index().groupby(['x','y']).category.value_counts().reset_index()
bmu_bins['cell']=bmu_bins.x.astype(str)+'_'+bmu_bins.y.astype(str)

In [8]:
bin_cats = pd.crosstab(
    index=bmu_bins.cell,
    columns=bmu_bins.category,
    values=bmu_bins['count'],
    aggfunc=sum).fillna(0)

In [9]:
bin_cats

category,adaptation,location,taxongroup,topic
cell,,,,
0_0,5.0,4.0,33.0,55.0
0_1,10.0,7.0,39.0,251.0
0_10,0.0,2.0,33.0,55.0
0_11,1.0,4.0,18.0,59.0
0_12,6.0,14.0,33.0,93.0
...,...,...,...,...
9_5,21.0,17.0,26.0,101.0
9_6,8.0,3.0,16.0,81.0
9_7,16.0,0.0,13.0,36.0


In [10]:
# information radius (JSD)
bin_cats.topic.describe()

count    320.000000
mean     123.540625
std       66.594677
min       26.000000
25%       86.750000
50%      115.000000
75%      143.000000
max      849.000000
Name: topic, dtype: float64

In [11]:
cat_bin_p = bin_cats / bin_cats.sum()

In [12]:
import scipy.special as spspec

In [14]:
irad_data = []
for cat1 in cat_bin_p.columns:
    for cat2 in cat_bin_p.columns:
        p1 = cat_bin_p[cat1]
        p2 = cat_bin_p[cat2]
        q = (p1+p2)/2
        kl_div1 = spspec.rel_entr(p1,q).sum()
        kl_div2 = spspec.rel_entr(p2,q).sum()
        inforad = (kl_div1+kl_div2) / 2

        row = (cat1,cat2,inforad)
        irad_data.append(row)

irad_data_df = pd.DataFrame(irad_data,columns=['cat1','cat2','IRad.JSD'])
irad_data_df
                           
        

,cat1,cat2,IRad.JSD
0,adaptation,adaptation,0.000000
1,adaptation,location,0.103116
2,adaptation,taxongroup,0.090545
3,adaptation,topic,0.084983
4,location,adaptation,0.103116
5,location,location,0.000000
6,location,taxongroup,0.099017
7,location,topic,0.076597
8,taxongroup,adaptation,0.090545
9,taxongroup,location,0.099017


In [15]:
irad_tbl_df = irad_data_df.pivot(columns='cat1',index='cat2')
irad_tbl_df

IRad.JSD                               
cat1       adaptation  location taxongroup     topic
cat2                                                
adaptation   0.000000  0.103116   0.090545  0.084983
location     0.103116  0.000000   0.099017  0.076597
taxongroup   0.090545  0.099017   0.000000  0.048041
topic        0.084983  0.076597   0.048041  0.000000

In [16]:
book_df = irad_tbl_df.round(3).astype(str)
print(book_df.to_latex())

\begin{tabular}{lllll}
\toprule
 & \multicolumn{4}{r}{IRad.JSD} \\
cat1 & adaptation & location & taxongroup & topic \\
cat2 &  &  &  &  \\
\midrule
adaptation & 0.0 & 0.103 & 0.091 & 0.085 \\
location & 0.103 & 0.0 & 0.099 & 0.077 \\
taxongroup & 0.091 & 0.099 & 0.0 & 0.048 \\
topic & 0.085 & 0.077 & 0.048 & 0.0 \\
\bottomrule
\end{tabular}



In [17]:
# import booktex

In [20]:
# The information radius for token categories, mapping to distinct SOM cells in the fitted 20x16 lattice.'
# [[tbl-8dim-ica-som-cat-irad-tex]]